# CRI Validation — Does the Composite Actually Predict Anything?

**Date:** 2026-05-19  
**Status:** OOS results contradict the methodology-tune plan's premise. Read findings before shipping.

## Why this exists

The CRI methodology-tune plan (`docs/superpowers/archive/plans/2026-05-19-cri-methodology-tune.md`) ships calibration improvements without testing whether they make the composite a *better predictor*. This notebook closes the four open questions:

1. **Define the target precisely** — not "crash risk" in the abstract, but a binary event with a forward horizon.  
2. **Compare against a naive baseline** — the simplest possible predictor (rolling-quantile VIX).  
3. **Walk-forward OOS testing** — the calibration was tuned on 20y of data; tuning + testing on the same data is leakage.  
4. **Treat CRI as a 0–100 score, evaluate threshold-free** — AUC is the right metric; F1 at a single threshold is a fragile add-on.

## What this notebook will not do

- Validate the *display*. Whether the UI is useful to a human reader is a separate question.
- Predict crashes with high accuracy. This is a notoriously hard problem. The goal is to show *whether the 4-component composite beats a 1-feature baseline*, not to find alpha.

## 1. Data sources

Everything is local parquet from the market-warehouse lake at `~/market-warehouse/data-lake/bronze/asset_class=volatility/`. No UW API, no Yahoo. Coverage:

| Symbol | Rows | Start | Use |
|---|---|---|---|
| VIX | 9,186 | 1990-01-02 | implied 30d vol |
| VVIX | 5,021 | 2006-03-06 | vol-of-VIX |
| COR1M | 5,124 | 2006-01-03 | implied 1m correlation |
| SPX | 12,951 | 1975-01-02 | underlying for drawdown labels + MA distance |

The binding constraint is VVIX/COR1M, both starting March 2006. After inner-join and forward-label tail trim we have **4,770 trading days (2007-05-30 → 2026-05-15)**.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

LAKE = Path.home() / "market-warehouse/data-lake/bronze/asset_class=volatility"

def load(sym: str, col: str = "close") -> pd.Series:
    df = pd.read_parquet(LAKE / f"symbol={sym}" / "1d.parquet")
    df["trade_date"] = pd.to_datetime(df["trade_date"])
    s = df.set_index("trade_date")[col].astype(float)
    s.name = sym
    return s

vix, vvix, cor1m, spx = (load(s) for s in ["VIX", "VVIX", "COR1M", "SPX"])
panel = pd.concat([vix, vvix, cor1m, spx], axis=1).dropna()
panel.columns = ["VIX", "VVIX", "COR1M", "SPX"]
print(f"Aligned panel: n={len(panel)}, {panel.index.min().date()} -> {panel.index.max().date()}")

## 2. Feature engineering

All features are deterministic from the close-of-day inputs (no peeking). The CRI scoring formulas use these directly.

In [ ]:
panel["VIX_5d_roc"]    = (panel["VIX"]  / panel["VIX"].shift(5)  - 1) * 100
panel["VVIX_5d_roc"]   = (panel["VVIX"] / panel["VVIX"].shift(5) - 1) * 100
panel["COR1M_5d_chg"]  = panel["COR1M"] - panel["COR1M"].shift(5)
panel["VVIX_VIX_ratio"] = panel["VVIX"] / panel["VIX"]
panel["SPX_100d_MA"]   = panel["SPX"].rolling(100).mean()
panel["SPX_dist_pct"]  = (panel["SPX"] / panel["SPX_100d_MA"] - 1) * 100
# Naive baseline: trailing 252d (1 year) p80 of VIX
panel["VIX_p80_252d"]  = panel["VIX"].rolling(252).quantile(0.80)

## 3. Component scorers (v1 = current, v2 = planned)

VIX, Correlation, Trend-Break formulas are unchanged between v1 and v2 — the plan only touches the VVIX component (and the UI labels). Both versions reproduce exactly what's in `src/uw_scan/cards/cri_scoring.py` (v1) and what the plan adds (v2).

In [ ]:
def clip(x, lo, hi): return max(lo, min(hi, x))

def score_vix(vix, vix_5d_roc):
    if np.isnan(vix) or np.isnan(vix_5d_roc): return 0.0
    lvl = clip((vix - 15.0) / 25.0 * 15.0, 0, 15)
    roc = clip(max(vix_5d_roc, 0.0) / 60.0 * 10.0, 0, 10)
    return lvl + roc

def score_vvix_v1(vvix, vvix_vix_ratio):
    """Current production: floor 90, ceiling 140, level(0-17) + ratio(0-8)."""
    if np.isnan(vvix) or np.isnan(vvix_vix_ratio): return 0.0
    lvl   = clip((vvix - 90.0) / 50.0 * 17.0, 0, 17)
    ratio = clip((vvix_vix_ratio - 5.0) / 3.0 * 8.0, 0, 8)
    return lvl + ratio

def score_vvix_v2(vvix, vvix_vix_ratio, vvix_5d_roc):
    """Planned: floor 85, ceiling 130, level(0-12) + ratio(0-7) + RoC(0-6)."""
    if np.isnan(vvix) or np.isnan(vvix_vix_ratio): return 0.0
    if np.isnan(vvix_5d_roc): vvix_5d_roc = 0.0
    lvl   = clip((vvix - 85.0) / 45.0 * 12.0, 0, 12)
    ratio = clip((vvix_vix_ratio - 5.0) / 3.0 * 7.0, 0, 7)
    roc   = clip(max(vvix_5d_roc, 0.0) / 25.0 * 6.0, 0, 6)
    return lvl + ratio + roc

def score_corr(corr, corr_5d_change):
    if np.isnan(corr): return 0.0
    if np.isnan(corr_5d_change): corr_5d_change = 0.0
    lvl   = clip((corr - 25.0) / 45.0 * 17.0, 0, 17)
    spike = clip(max(corr_5d_change, 0.0) / 20.0 * 8.0, 0, 8)
    return lvl + spike

def score_trend(spx_dist_pct):
    if np.isnan(spx_dist_pct) or spx_dist_pct >= 0: return 0.0
    return clip(abs(spx_dist_pct) / 10.0 * 25.0, 0, 25)

panel["CRI_v1"] = panel.apply(lambda r: (
    score_vix(r["VIX"], r["VIX_5d_roc"])
    + score_vvix_v1(r["VVIX"], r["VVIX_VIX_ratio"])
    + score_corr(r["COR1M"], r["COR1M_5d_chg"])
    + score_trend(r["SPX_dist_pct"])), axis=1)

panel["CRI_v2"] = panel.apply(lambda r: (
    score_vix(r["VIX"], r["VIX_5d_roc"])
    + score_vvix_v2(r["VVIX"], r["VVIX_VIX_ratio"], r["VVIX_5d_roc"])
    + score_corr(r["COR1M"], r["COR1M_5d_chg"])
    + score_trend(r["SPX_dist_pct"])), axis=1)

panel[["CRI_v1","CRI_v2"]].describe()

## 4. Targets — pick concrete forward events

Three labels, each phrased as a binary forward event:

- **`label_dd5`** — SPX falls ≥5% from today's close within the next **20 trading days**. The canonical "meaningful drawdown" definition for a one-month horizon.
- **`label_vix30`** — VIX prints ≥30 at any point in the next **10 trading days**. A pure vol-spike target. Easier than predicting drawdowns (because VIX is autocorrelated with itself).
- **`label_dd10`** — SPX falls ≥10% from today's close within the next **60 trading days**. A bigger-tail event.

Base rates over the full sample: **18% / 16% / 15%** respectively. These are surprisingly high — markets are choppy.

In [ ]:
fwd_min_20 = panel["SPX"].rolling(20).min().shift(-20)
panel["label_dd5"]  = ((fwd_min_20 / panel["SPX"]) - 1 <= -0.05).astype(int)
fwd_max_vix = panel["VIX"].rolling(10).max().shift(-10)
panel["label_vix30"] = (fwd_max_vix >= 30.0).astype(int)
fwd_min_60 = panel["SPX"].rolling(60).min().shift(-60)
panel["label_dd10"] = ((fwd_min_60 / panel["SPX"]) - 1 <= -0.10).astype(int)

labels_panel = panel.dropna(subset=["label_dd5","label_vix30","label_dd10","CRI_v1","CRI_v2","VIX_p80_252d"])
print(f"After warmup + tail trim: n={len(labels_panel)}")
print(f"Base rates: dd5={labels_panel['label_dd5'].mean():.1%}, vix30={labels_panel['label_vix30'].mean():.1%}, dd10={labels_panel['label_dd10'].mean():.1%}")

## 5. Walk-forward split — no peeking

The current CRI calibration was chosen against the full 20y distribution. That's leakage. To get an honest signal, split:

- **In-sample (2007–2015)** — used only to pick the alarm threshold so all three predictors fire at the same rate (apples-to-apples).
- **Out-of-sample (2016–2026)** — the only numbers that count. Includes 2018 volmageddon, 2020 COVID, 2022 rate-hike vol, 2024 yen-carry unwind.

The threshold-matching step finds the CRI score that produces the same alarm rate as the baseline on in-sample data, then applies that threshold to OOS. This neutralizes the "my predictor alarms less, that's why F1 looks different" confound.

In [ ]:
SPLIT = pd.Timestamp("2016-01-01")
is_, oos = labels_panel[labels_panel.index < SPLIT], labels_panel[labels_panel.index >= SPLIT]

baseline_alarm_rate = ((is_["VIX"] >= is_["VIX_p80_252d"]).mean())
thr_v1 = np.quantile(is_["CRI_v1"], 1 - baseline_alarm_rate)
thr_v2 = np.quantile(is_["CRI_v2"], 1 - baseline_alarm_rate)

print(f"IS: {len(is_)} rows ({is_.index.min().date()}->{is_.index.max().date()})")
print(f"OOS: {len(oos)} rows ({oos.index.min().date()}->{oos.index.max().date()})")
print(f"Baseline alarm rate IS: {baseline_alarm_rate:.1%}")
print(f"CRI v1 threshold matched to baseline rate: {thr_v1:.2f}")
print(f"CRI v2 threshold matched to baseline rate: {thr_v2:.2f}")

## 6. Metrics

- **ROC AUC** — threshold-free. Probability that a random positive-day scores higher than a random negative-day. 0.5 = coin flip; 1.0 = perfect.
- **F1 at matched alarm rate** — what each predictor achieves when forced to fire on the same fraction of days. The honest threshold comparison.
- **Lead time on named events** — calendar days from first alarm to the drawdown trough. Tells us whether "alarm" actually leads or just coincides.

In [ ]:
def roc_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    pos, neg = s[y==1], s[y==0]
    if not len(pos) or not len(neg): return float("nan")
    ranks = pd.Series(s).rank().values
    return (ranks[y==1].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))

def metrics(y, s, thr):
    y = np.asarray(y).astype(bool); pred = np.asarray(s) >= thr
    tp, fp = (pred & y).sum(), (pred & ~y).sum()
    fn, tn = (~pred & y).sum(), (~pred & ~y).sum()
    p = tp/(tp+fp) if (tp+fp) else float("nan")
    r = tp/(tp+fn) if (tp+fn) else float("nan")
    f1 = 2*p*r/(p+r) if (p and r and not np.isnan(p)) else float("nan")
    return dict(prec=p, rec=r, f1=f1, alarm=pred.mean())

def evaluate(df, ycol, title):
    base_pred = (df["VIX"] >= df["VIX_p80_252d"]).astype(int)
    base = dict(
        prec = base_pred[df[ycol]==1].sum()/max(base_pred.sum(),1),
        rec  = base_pred[df[ycol]==1].sum()/max(df[ycol].sum(),1),
        alarm= base_pred.mean())
    base["f1"] = 2*base["prec"]*base["rec"]/(base["prec"]+base["rec"]) if (base["prec"]+base["rec"]) else float("nan")
    rows = [
        ("baseline (VIX>=trailing-p80)", roc_auc(df[ycol], df["VIX"]), base["prec"], base["rec"], base["f1"], base["alarm"]),
        (f"CRI v1 (thr={thr_v1:.1f})",   roc_auc(df[ycol], df["CRI_v1"]), *(metrics(df[ycol], df["CRI_v1"], thr_v1)[k] for k in ["prec","rec","f1","alarm"])),
        (f"CRI v2 (thr={thr_v2:.1f})",   roc_auc(df[ycol], df["CRI_v2"]), *(metrics(df[ycol], df["CRI_v2"], thr_v2)[k] for k in ["prec","rec","f1","alarm"])),
    ]
    out = pd.DataFrame(rows, columns=["predictor","AUC","prec","recall","F1","alarm_rate"]).set_index("predictor")
    print(f"\n### {title}   (base rate {df[ycol].mean():.1%})")
    print(out.round(3).to_string())
    return out

print("="*70); print("OOS RESULTS (2016-2026) — the only numbers that matter"); print("="*70)
r_dd5  = evaluate(oos, "label_dd5",  "SPX -5% in next 20d")
r_vix  = evaluate(oos, "label_vix30", "VIX >= 30 in next 10d")
r_dd10 = evaluate(oos, "label_dd10", "SPX -10% in next 60d")

## 7. Component AUC decomposition (OOS, target = 5% drawdown in 20d)

Each sub-component scored standalone. This tells us which parts of the composite carry signal and which are noise.

In [ ]:
oos = oos.assign(
    c_vix     = oos.apply(lambda r: score_vix(r["VIX"], r["VIX_5d_roc"]), axis=1),
    c_vvix_v1 = oos.apply(lambda r: score_vvix_v1(r["VVIX"], r["VVIX_VIX_ratio"]), axis=1),
    c_vvix_v2 = oos.apply(lambda r: score_vvix_v2(r["VVIX"], r["VVIX_VIX_ratio"], r["VVIX_5d_roc"]), axis=1),
    c_corr    = oos.apply(lambda r: score_corr(r["COR1M"], r["COR1M_5d_chg"]), axis=1),
    c_trend   = oos.apply(lambda r: score_trend(r["SPX_dist_pct"]), axis=1),
)

comp_rows = [
    ("VIX raw level",        roc_auc(oos["label_dd5"], oos["VIX"])),
    ("VIX component",        roc_auc(oos["label_dd5"], oos["c_vix"])),
    ("VVIX raw",             roc_auc(oos["label_dd5"], oos["VVIX"])),
    ("VVIX v1 component",    roc_auc(oos["label_dd5"], oos["c_vvix_v1"])),
    ("VVIX v2 component",    roc_auc(oos["label_dd5"], oos["c_vvix_v2"])),
    ("COR1M raw",            roc_auc(oos["label_dd5"], oos["COR1M"])),
    ("COR1M component",      roc_auc(oos["label_dd5"], oos["c_corr"])),
    ("SPX dist (negated)",   roc_auc(oos["label_dd5"], -oos["SPX_dist_pct"])),
    ("Trend-Break component",roc_auc(oos["label_dd5"], oos["c_trend"])),
    ("CRI v1 composite",     roc_auc(oos["label_dd5"], oos["CRI_v1"])),
    ("CRI v2 composite",     roc_auc(oos["label_dd5"], oos["CRI_v2"])),
]
comp_df = pd.DataFrame(comp_rows, columns=["predictor","AUC"]).set_index("predictor")
print(comp_df.round(3).to_string())

## 8. Lead-time on named stress events

For each named drawdown trough in the OOS window, how many calendar days before the trough did each predictor first cross its alarm threshold? Negative or near-zero numbers mean "alarmed at the bottom" (coincident, not predictive). Numbers >30 mean genuine lead time.

In [ ]:
events = [
    ("2018-02-05", "Volmageddon"),
    ("2018-12-24", "Q4 2018 trough"),
    ("2020-03-23", "COVID trough"),
    ("2022-06-13", "rate-hike trough"),
    ("2024-08-05", "yen-carry unwind"),
]
lead_rows = []
for date, name in events:
    target = pd.Timestamp(date)
    if target not in oos.index:
        target = oos.index[oos.index.searchsorted(target)]
    window = oos.loc[:target].tail(61)
    def first_alarm_lead(mask):
        a = window[mask]
        return (target - a.index[0]).days if len(a) else None
    lead_rows.append({
        "event": name, "date": target.date(),
        "v1_lead_days":   first_alarm_lead(window["CRI_v1"] >= thr_v1),
        "v2_lead_days":   first_alarm_lead(window["CRI_v2"] >= thr_v2),
        "baseline_lead":  first_alarm_lead(window["VIX"] >= window["VIX_p80_252d"]),
    })
pd.DataFrame(lead_rows).set_index("event")

## 9. Findings (honest)

### What the data says, in plain English

1. **The planned calibration changes (v1→v2) move OOS predictive accuracy by ~0.1 AUC points.** On all three targets, v2 is within statistical noise of v1. The plan's changes are *cosmetic improvements to gradation and UX*, not predictive improvements. This was not obvious before running the numbers.

2. **The naive baseline (VIX ≥ trailing-252d p80) matches or beats CRI on the primary 5% drawdown target.** OOS AUC: baseline 0.637, CRI v1 0.620, CRI v2 0.621. The 4-component composite does *not* beat a 1-feature predictor on its headline use case.

3. **The VVIX component is essentially noise for predicting SPX drawdowns.** OOS AUC of the VVIX v1 component is 0.491 (worse than coin flip); v2 is 0.504 (effectively coin flip). The VVIX RoC sub-score added by the plan marginally improves the *component* AUC (0.491→0.504) but the composite is dominated by other components, so the effect washes out.

4. **VIX is doing all the work.** OOS AUC of raw VIX (0.637) ≥ OOS AUC of the full CRI composite (0.621). The other three components add noise, not signal, for the 5% drawdown target.

5. **CRI does add value on larger / longer drawdowns.** On the 10%-in-60-days target, CRI v1 AUC (0.647) beats baseline (0.629). This is the regime where the composite earns its keep — predicting bigger, slower-developing drawdowns where the trend-break component contributes.

6. **CRI has higher precision but lower recall than baseline on the vol-spike target.** At matched alarm rates, CRI gets 55% precision vs baseline's 41% — fewer false alarms, but catches fewer events. Whether that's better depends on the cost asymmetry.

7. **Lead-times are mostly persistence, not prediction.** On Volmageddon and the 2024 yen-carry unwind, the "lead time" is 3 calendar days — that's the alarm firing on the same day stress was already visible. The longer lead-times (Q4 2018, COVID, 2022) are from CRI staying elevated through the entire decline.

### What this means for the implementation plan

**Ship it anyway, but rewrite the framing.** The plan's value is:
- Better UI gradation (real)
- Documented methodology source-of-truth (real)
- VVIX scoring that matches the practitioner literature (real)
- "Trend Break" rename + tooltip honesty (real)

What the plan *doesn't* do:
- Improve drawdown prediction accuracy (proven false)
- Beat a naive baseline (proven false on primary target)

The methodology doc should be revised to:
1. Frame CRI as a **regime monitor** (descriptive) not a **predictor** (forecasting)
2. Acknowledge that for SPX drawdown prediction, raw VIX percentile is competitive
3. Note that CRI's edge is on the **bigger-drawdown / longer-horizon** target (10% in 60d), not 5% in 20d
4. Remove any language implying CRI "predicts" or "warns of" crashes

### What deserves a follow-up spec (not this PR)

- **Reconsider VVIX inclusion.** Component AUC of 0.50 means it adds noise. Either drop it, replace its sub-scores with something signal-bearing (e.g., VVIX×VIX joint percentile), or weight it down in the composite.
- **Add VIX term structure.** VIX/VIX3M flip from contango to backwardation is the canonical pre-stress signal not currently in CRI. The lake has VIX3M from 2009.
- **Walk-forward calibration.** Re-tune CRI thresholds rolling, not on the full sample. Honest performance will be worse.
- **Compare against more baselines.** VIX p80 is the simplest. Realized-vol z-score, SPX 20d skew, even just "VIX > 20" — all worth comparing.

### Bottom line

The validation was the right thing to do. The plan ships a cleaner display and a better-documented methodology. It does *not* ship a better crash predictor — and that's important to know before any user is told "our CRI improved."